In [0]:
from pyspark.sql.functions import (
    col,
    sum,
    countDistinct,
    count,
    round,
    min,
    max,
    avg,
    current_timestamp
)

CATALOG = "dbw_ecommerce_om"

SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

print("Gold analytics configuration loaded")
print(f"Source: {CATALOG}.{SILVER_SCHEMA}")
print(f"Target: {CATALOG}.{GOLD_SCHEMA}")

Gold analytics configuration loaded
Source: dbw_ecommerce_om.silver
Target: dbw_ecommerce_om.gold


In [0]:
silver_customers = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.customers"
)

silver_products = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.products"
)

silver_orders = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.orders"
)

silver_order_items = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.order_items"
)

silver_payments = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.payments"
)

print("All five Silver tables loaded")

All five Silver tables loaded


In [0]:
daily_sales = (
    silver_order_items
    .join(
        silver_orders.select(
            "order_id",
            "order_date"
        ),
        on="order_id",
        how="inner"
    )
    .withColumn(
        "sales_amount",
        col("quantity") * col("price")
    )
    .groupBy("order_date")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum("sales_amount"), 2).alias("total_sales")
    )
    .orderBy("order_date")
)

print("Daily sales transformation created")

Daily sales transformation created


In [0]:
customer_sales = (
    silver_orders
    .join(
        silver_order_items,
        on="order_id",
        how="inner"
    )
    .groupBy("customer_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(
            sum(col("quantity") * col("price")),
            2
        ).alias("total_sales"),
        round(
            avg(col("quantity") * col("price")),
            2
        ).alias("average_order_value")
    )
    .orderBy(col("total_sales").desc())
)

print("Customer sales transformation created")

Customer sales transformation created


In [0]:
product_sales = (
    silver_order_items
    .join(
        silver_products.select(
            "product_id",
            "product_name",
            "category"
        ),
        on="product_id",
        how="inner"
    )
    .groupBy(
        "product_id",
        "product_name",
        "category"
    )
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(
            sum(col("quantity") * col("price")),
            2
        ).alias("total_sales")
    )
    .orderBy(col("total_sales").desc())
)

print("Product sales transformation created")

Product sales transformation created


In [0]:
category_sales = (
    silver_order_items
    .join(
        silver_products.select(
            "product_id",
            "category"
        ),
        on="product_id",
        how="inner"
    )
    .groupBy("category")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(
            sum(col("quantity") * col("price")),
            2
        ).alias("total_sales")
    )
    .orderBy(col("total_sales").desc())
)

print("Category sales transformation created")

Category sales transformation created


In [0]:
sales_summary = (
    silver_order_items
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(
            sum(col("quantity") * col("price")),
            2
        ).alias("total_sales"),
        round(
            avg(col("quantity") * col("price")),
            2
        ).alias("average_line_item_value")
    )
)

print("Overall sales summary transformation created")

Overall sales summary transformation created


In [0]:
gold_tables = {
    "daily_sales": daily_sales,
    "customer_sales": customer_sales,
    "product_sales": product_sales,
    "category_sales": category_sales,
    "sales_summary": sales_summary
}

for table_name, df in gold_tables.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
        )
    )

    print(
        f"Written: {CATALOG}.{GOLD_SCHEMA}.{table_name}"
    )

print("All Gold tables written successfully")

Written: dbw_ecommerce_om.gold.daily_sales
Written: dbw_ecommerce_om.gold.customer_sales
Written: dbw_ecommerce_om.gold.product_sales
Written: dbw_ecommerce_om.gold.category_sales
Written: dbw_ecommerce_om.gold.sales_summary
All Gold tables written successfully


In [0]:
print("========== GOLD VALIDATION ==========")

for table_name in gold_tables:
    df = spark.table(
        f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
    )

    print(f"{table_name}: {df.count()} rows")
    print(f"Columns: {df.columns}")
    print("-" * 60)


print("========== SALES SUMMARY ==========")

spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.sales_summary"
).show(truncate=False)


print("========== CATEGORY SALES ==========")

spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.category_sales"
).show(truncate=False)


print("Gold validation completed successfully")

========== GOLD VALIDATION ==========
daily_sales: 181 rows
Columns: ['order_date', 'total_orders', 'total_quantity', 'total_sales']
------------------------------------------------------------
customer_sales: 9999 rows
Columns: ['customer_id', 'total_orders', 'total_quantity', 'total_sales', 'average_order_value']
------------------------------------------------------------
product_sales: 2000 rows
Columns: ['product_id', 'product_name', 'category', 'total_orders', 'total_quantity', 'total_sales']
------------------------------------------------------------
category_sales: 8 rows
Columns: ['category', 'total_orders', 'total_quantity', 'total_sales']
------------------------------------------------------------
sales_summary: 1 rows
Columns: ['total_orders', 'total_quantity', 'total_sales', 'average_line_item_value']
------------------------------------------------------------
========== SALES SUMMARY ==========
+------------+--------------+---------------+-----------------------+
|tota